In [1]:
def triangleFinite(w,m1,m2,m3):
    if (w^m1).is_one() or (w^m2).is_one() or (w^m3).is_one():
        return True
    else:
        return False

In [2]:
#checks is r1 seperates r2 from the identity, a wall seperates itself from the identity
def seperateFromOne(r1,r2,m1,m2,m3):
    r=r1*r2
    if (r1*(r2^-1)).is_one():
        return True
    if triangleFinite(r,m1,m2,m3):
        return False
    elif (r).length()<(r2).length():
        return True
    else:
        return False

In [3]:
def wallsOfOne(G,m1,m2,m3):
    #fin is the indicator for the process of finding walls, parallel is a statement about the current wall being checked
    fin=False
    parallel=False
    #Initializing the list with the walls of the fundamental chamber, so start check at walls gsg^-1 where g has length 1
    n=1
    s=G.simple_reflections()
    H=[]
    for t in s:
        H.append(t.reduced_word())
    while fin==False:
        #if we find any new walls we stop, if we find a new wall we flip this
        fin=True
        #looking at walls at distance n from the identity
        for g in list(G.elements_of_length(n)):
            #got 3 reflections around any point, redcing this to 2 is possible
            
            for t in s:
                #we assume the new wall is not seperated from 1 until we find a wall in H that does
                parallel=False
                #the reflection we are checking
                gRef=g*t*(g^-1)
                #print(gRef.reduced_word())
                #checking if it is a new one
                for rl in H:
                    #convert the reduced word to group element
                    r=G.from_reduced_word(rl)
                    #check if g is seperated from the unit by this wall, if it is seperated we can ignore it
                    if seperateFromOne(r,gRef,m1,m2,m3):
                            parallel=True
                            #print("seperated by ")
                            #print(rl)
                            #print(" ")
                #if we found a new wall add (a reduced word for it) to the list
                #also need to look a layer deeper
                if parallel==False:
                    H.append(gRef.reduced_word())
                    fin=False
        print(n)
        #increase the search radius
        n=n+1
    return H, n
                

In [97]:
#reflections seerating an element from the unit bring it closer
def seperateFromOneElem(r,g):
    if (r*g).length()<g.length():
        return True
    else:
        return False

In [105]:
#find all the walls seperating g not seperated from g by another wall
def wallsOfElem(G,g,m1,m2,m3,n):
    #all walls seperating g from e are represented in (any) shortest representative
    l=g.length()
    w=g.reduced_word()
    #closest one is definitely one of them
    h=G.from_reduced_word(w[:l-1])
    r=G.from_reduced_word([w[l-1]])
    ref=h*r*(h^-1)
    Wg=[]
    Wg.append(ref.reduced_word())
    #work down, knowing the radius of the walls seperating the identity gives a bound, might be n-1, but I want to be safe
    for i in range(n+1):
        #check if seperated by one closer
        parallel=False
        #underflow issues
        k=max(l-i-1,0)
        #build reflection
        h=G.from_reduced_word(w[:k])
        r=G.from_reduced_word([w[k]])
        ref=h*r*(h^-1)
        #check if it is a new one
        for rl in Wg:
            r2=G.from_reduced_word(rl)
            #check if we should not add
            if seperateFromOne(ref,r2,m1,m2,m3):
                parallel=True
        if parallel==False:
            Wg.append(ref.reduced_word())
    return Wg
        

In [106]:
def voraciousProj(G,g,m1,m2,m3,n):
    Wg=wallsOfElem(G,g,m1,m2,m3,n)
    lmin=len(Wg)
    w=g.reduced_word()
    l=len(w)
    #the voracious projection is passed by all geodesics, so after picking one we just go back until we pass all the walls
    #pass a minimum of |Wg| of them 
    for i in range(lmin-1,l+1):
        seperate=True
        p=G.from_reduced_word(w[:l-i])
        for rl in Wg:
            r=G.from_reduced_word(rl)
            if seperateFromOneElem(r,p):
                seperate=False
        if seperate==True:
            return p
    return g

In [7]:
M=matrix([[1,3,7],[3,1,3],[7,3,1]])

In [8]:
W=CoxeterGroup(M)

In [9]:
W

Coxeter group over Universal Cyclotomic Field with Coxeter matrix:
[1 3 7]
[3 1 3]
[7 3 1]

In [10]:
s=W.simple_reflections()

In [11]:
w=s[3]*s[2]*s[3]*s[1]

In [27]:
w1=w.reduced_word()

In [30]:
w1

[2, 3, 2, 1]

In [40]:
h=W.from_reduced_word(w1[:4-1-1])
r=W.from_reduced_word([w1[4-1-1]])
ref=h*r*(h^-1)

In [69]:
r.reduced_word()

[2]

In [42]:
seperateFromOne(ref,W.from_reduced_word([2, 3, 1, 2, 1, 3, 2]),3,3,7)

False

In [109]:
Wg=wallsOfElem(W,w,3,3,7,5)

In [110]:
Wg

[[2, 3, 1, 2, 1, 3, 2], [3], [2]]

In [107]:
wallsOfElem(W,W.from_reduced_word([2,3,1]),3,3,7,5)

[[2, 3, 1, 3, 2], [2, 3, 2]]

In [108]:
voraciousProj(W,W.from_reduced_word([2,3,1]),3,3,7,5).reduced_word()

[2]

In [83]:
h=W.from_reduced_word([2,3])
r=W.from_reduced_word([2])
ref=h*r*(h^-1)

In [84]:
ref.reduced_word()

[3]

In [10]:
H, n =wallsOfOne(W,3,3,7)

1
2
3
4


In [11]:
H

[[1],
 [2],
 [3],
 [1, 2, 1],
 [1, 3, 1],
 [2, 3, 2],
 [3, 1, 3],
 [1, 3, 1, 3, 1],
 [3, 1, 3, 1, 3],
 [1, 3, 1, 3, 1, 3, 1]]

In [12]:
n

5